In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("movie_metadata.csv")

In [3]:
# Clean columns
df["genres"] = df["genres"].astype(str)
df["movie_title"] = df["movie_title"].astype(str)
df["actor_1_name"] = df["actor_1_name"].astype(str)
df["actor_2_name"] = df["actor_2_name"].astype(str)
df["actor_3_name"] = df["actor_3_name"].astype(str)


In [5]:
def detect_mood(text):
    text = text.lower()

    mood_keywords = {
        "happy": ["happy", "joy", "excited", "glad"],
        "sad": ["sad", "depressed", "down"],
        "angry": ["angry", "mad", "furious"],
        "romantic": ["romantic", "love"],
        "lonely": ["lonely", "alone", "isolated"]
    }

    for mood, words in mood_keywords.items():
        for w in words:
            if w in text:
                return mood

    return "neutral"

In [7]:
mood_to_genre = {
    "happy": ["Comedy", "Adventure", "Family"],
    "sad": ["Drama", "Biography"],
    "angry": ["Action", "Thriller"],
    "romantic": ["Romance", "Drama"],
    "lonely": ["Family", "Drama", "Friendship"],
    "neutral": ["Drama"]
}

In [9]:
def recommend_movies(text):

    text_low = text.lower()
    mood = detect_mood(text)

    # ------------------------------------------
    # CASE 1: INPUT IS A MOOD SENTENCE
    # ------------------------------------------
    if mood is not None:
        genres = mood_to_genre[mood]
        print("Detected Mood →", mood)
        print("Mapped Genres →", genres)

        filtered = df[df["genres"].str.contains("|".join(genres), case=False, na=False)]
        filtered = filtered.sort_values(by="imdb_score", ascending=False)

        return filtered["movie_title"].head(5).tolist()

    # ------------------------------------------
    # CASE 2: INPUT IS ACTOR / ACTRESS NAME
    # ------------------------------------------
    else:
        print("Detected Actor →", text)

        filtered = df[
            df["actor_1_name"].str.lower().str.contains(text_low) |
            df["actor_2_name"].str.lower().str.contains(text_low) |
            df["actor_3_name"].str.lower().str.contains(text_low)
        ]

        if filtered.empty:
            return ["No movies found for this actor"]

        filtered = filtered.sort_values(by="imdb_score", ascending=False)

        return filtered["movie_title"].head(5).tolist()


In [11]:
print("Mood Input Test:")
print(recommend_movies("I am feeling happy"))

print("\nActor Input Test:")
print(recommend_movies("Leonardo DiCaprio"))

Mood Input Test:
Detected Mood → happy
Mapped Genres → ['Comedy', 'Adventure', 'Family']
['Towering Inferno\xa0            ', 'The Lord of the Rings: The Return of the King\xa0', 'Inception\xa0', 'Star Wars: Episode V - The Empire Strikes Back\xa0', "It's Always Sunny in Philadelphia\xa0            "]

Actor Input Test:
Detected Mood → neutral
Mapped Genres → ['Drama']
['The Shawshank Redemption\xa0', 'The Godfather\xa0', 'Dekalog\xa0            ', 'Dekalog\xa0            ', 'The Dark Knight\xa0']
